# PS S6E7 — V2: HistGradientBoosting + exact-value TargetEncoder (5-seed)

**전략 요약**
- 라벨은 수면·스트레스·활동 3요인의 준-결정적 규칙 + 합성 노이즈 → 표현(representation)이 병목
- 핵심 기법: **수치형 exact-value TargetEncoder** (각 정확한 값 = 카테고리, fold 내 교차적합) + HistGB
- 규칙 피처(sleep<6/<7, rule_pred, 결측지시자) + 서수 인코딩
- 층화 5-fold × 5시드 평균, `class_weight='balanced'`
- 로컬 검증: OOF Balanced Accuracy **0.95051** (V1 0.94962 → +0.0009)

과제 준수: scikit-learn API only · 층화추출 · 불균형 가중 · OOF 기준 판단

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score

DATA = '/kaggle/input/playground-series-s6e7'
CLASSES = ['at-risk', 'unhealthy', 'fit']
N_CLASS = 3
SEEDS = [42, 2026, 7, 101, 777]
N_SPLITS = 5

train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
mapping = {c: i for i, c in enumerate(CLASSES)}
y = train['health_condition'].map(mapping).to_numpy()
print(train.shape, test.shape, np.bincount(y))

In [ ]:
NUM = ['sleep_duration','heart_rate','bmi','calorie_expenditure',
       'step_count','exercise_duration','water_intake']
ORDINAL = {
    'stress_level':            {'low':0,'medium':1,'high':2},
    'sleep_quality':           {'poor':0,'average':1,'good':2},
    'physical_activity_level': {'sedentary':0,'moderate':1,'active':2},
    'smoking_alcohol':         {'no':0,'occasional':1,'yes':2},
    'diet_type':               {'veg':0,'balanced':1,'non-veg':2},
    'gender':                  {'female':0,'male':1,'other':2},
}
CAT = list(ORDINAL)

def encode(df):
    X = df[NUM + CAT].copy()
    for c, m in ORDINAL.items():
        X[c] = X[c].map(m).astype('float64')
    # 규칙 피처 (역공학된 생성규칙 기반, 결측은 NaN 유지)
    s, st, act = X['sleep_duration'], X['stress_level'], X['physical_activity_level']
    X['sleep_lt6'] = np.where(s.isna(), np.nan, (s < 6).astype(float))
    X['sleep_lt7'] = np.where(s.isna(), np.nan, (s < 7).astype(float))
    rp = np.full(len(X), np.nan)
    known = ~(s.isna() | st.isna() | act.isna())
    sv, stv, av = s[known], st[known], act[known]
    r = np.zeros(known.sum())
    r[(sv < 6) & (stv == 2)] = 1.0
    r[(sv >= 7) & (stv == 0) & (av == 2)] = 2.0
    rp[known.to_numpy()] = r
    X['rule_pred'] = rp
    X['miss_sleep'] = s.isna().astype(float)
    X['miss_stress'] = st.isna().astype(float)
    X['miss_activity'] = act.isna().astype(float)
    return X

X_all, X_test_all = encode(train), encode(test)
print(X_all.shape, X_test_all.shape)

In [ ]:
def run_seed(seed):
    oof = np.zeros((len(X_all), N_CLASS))
    test_pred = np.zeros((len(X_test_all), N_CLASS))
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)  # 층화추출
    for fold, (tr, va) in enumerate(skf.split(X_all, y)):
        X_tr, X_va, X_te = X_all.iloc[tr].copy(), X_all.iloc[va].copy(), X_test_all.copy()
        y_tr = y[tr]
        # 수치형 exact-value TE (fold 내부 교차적합 -> 누수 없음)
        te = TargetEncoder(target_type='multiclass', cv=5, random_state=seed)
        te_tr = te.fit_transform(X_tr[NUM].fillna(-999.0), y_tr)
        te_va = te.transform(X_va[NUM].fillna(-999.0))
        te_te = te.transform(X_te[NUM].fillna(-999.0))
        cols = [f'te_{c}_{k}' for c in NUM for k in range(N_CLASS)]
        X_tr = pd.concat([X_tr.reset_index(drop=True), pd.DataFrame(te_tr, columns=cols)], axis=1)
        X_va = pd.concat([X_va.reset_index(drop=True), pd.DataFrame(te_va, columns=cols)], axis=1)
        X_te = pd.concat([X_te.reset_index(drop=True), pd.DataFrame(te_te, columns=cols)], axis=1)
        model = HistGradientBoostingClassifier(
            max_iter=2000, learning_rate=0.05, max_leaf_nodes=63,
            early_stopping=True, validation_fraction=0.08, n_iter_no_change=50,  # valid 모니터링
            class_weight='balanced', random_state=seed + fold)
        model.fit(X_tr, y_tr)
        oof[va] = model.predict_proba(X_va)
        test_pred += model.predict_proba(X_te) / N_SPLITS
        print(f'  seed{seed} fold{fold} bal_acc={balanced_accuracy_score(y[va], oof[va].argmax(1)):.5f}')
    cv = balanced_accuracy_score(y, oof.argmax(1))
    print(f'[seed {seed}] OOF={cv:.5f}')
    return oof, test_pred

t0 = time.time()
oof_sum = np.zeros((len(X_all), N_CLASS))
test_sum = np.zeros((len(X_test_all), N_CLASS))
for sd in SEEDS:
    o, t = run_seed(sd)
    oof_sum += o / len(SEEDS)
    test_sum += t / len(SEEDS)
print(f'total {time.time()-t0:.0f}s')

In [ ]:
cv = balanced_accuracy_score(y, oof_sum.argmax(1))
print(f'V2 blend x{len(SEEDS)} seeds — OOF balanced_accuracy = {cv:.5f}')

inv = {i: c for i, c in enumerate(CLASSES)}
sub = pd.DataFrame({'id': test['id'],
                    'health_condition': [inv[i] for i in test_sum.argmax(1)]})
sub.to_csv('submission.csv', index=False)

ss = pd.read_csv(f'{DATA}/sample_submission.csv')
assert len(sub) == len(ss) and list(sub.columns) == list(ss.columns)
assert sub['id'].tolist() == ss['id'].tolist()
assert set(sub['health_condition']) <= set(CLASSES)
print('submission.csv saved | sanity OK |', sub['health_condition'].value_counts().to_dict())

## 실험 로그 (로컬 검증 기록)

| 실험 | 변경점 | OOF (bal_acc) | 판정 |
|---|---|---|---|
| V1 | LGBM 5-fold, class_weight balanced | 0.94962 | 기준점 (LB 0.94952, 갭 0.0001) |
| V2-A | HistGB + 서수 인코딩 | 0.94945 | V1 동급, 3배 빠름 |
| V2-B | + 규칙 피처 | 0.94950 | +0.00005 노이즈 수준 |
| V2-C | + 수치형 exact-value TE | 0.95028 | **+0.0008 실질 개선** |
| V2-D | + 원본 50k 증강 | 0.94970 | -0.0006 악화, 기각 |
| **V2-C×5** | 5시드 평균 | **0.95051** | 최종 채택 |